# Understanding Generation and Citations

This notebook is a learning surface for saved PubMedQA and SciFact generation artifacts.

It does not call the model API, rerun retrieval, or rewrite experiment outputs. It only reads JSONL artifacts and summary JSON files that were already produced by the runners and evaluator.

## 1. Setup

Run from the `rag_project/` repo root. The paths below point at the dense v4 n=100 generated artifacts.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()

ARTIFACTS = {
    'pubmedqa': PROJECT_ROOT / 'outputs/generation/pubmedqa_dense_v4_generated_n100_v01.jsonl',
    'scifact': PROJECT_ROOT / 'outputs/generation/scifact_dense_v4_generated_n100_v01.jsonl',
}

SUMMARIES = {
    'pubmedqa': PROJECT_ROOT / 'outputs/evaluation/pubmedqa_dense_v4_generated_n100_summary.json',
    'scifact': PROJECT_ROOT / 'outputs/evaluation/scifact_dense_v4_generated_n100_summary.json',
}

def load_jsonl(path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def load_json(path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

records = {dataset: load_jsonl(path) for dataset, path in ARTIFACTS.items()}
summaries = {dataset: load_json(path) for dataset, path in SUMMARIES.items()}

pd.DataFrame([
    {
        'dataset': dataset,
        'artifact_exists': ARTIFACTS[dataset].exists(),
        'summary_exists': SUMMARIES[dataset].exists(),
        'rows': len(records[dataset]),
    }
    for dataset in ARTIFACTS
])

## 2. Summary Metrics

The evaluator keeps retrieval metrics, answer metrics, generation errors, and citation metrics in the same summary object.

In [ ]:
def metric_rows(summaries):
    selected = [
        'answer_accuracy',
        'label_accuracy',
        'context_hit_at_k',
        'evidence_sentence_hit_at_k',
        'context_recall_at_k',
        'evidence_recall_at_k',
        'citation_valid_rate',
        'citation_no_citation_rate',
        'cited_passage_valid_rate',
        'citation_gold_hit_rate',
        'citation_gold_recall',
    ]
    rows = []
    for dataset, summary in summaries.items():
        metrics = summary['metrics']
        for key in selected:
            if key in metrics:
                rows.append({'dataset': dataset, 'metric': key, 'value': metrics[key]})
        rows.append({'dataset': dataset, 'metric': 'generation_error_count', 'value': summary['generation_error_count']})
    return pd.DataFrame(rows)

display(metric_rows(summaries).pivot(index='metric', columns='dataset', values='value'))

## 3. Per-Example Tags

These helpers turn each artifact row into compact tags for answer correctness, retrieval support, citation validity, and citation overlap with gold evidence.

In [ ]:
def normalize_answer(dataset, value):
    text = str(value or '').strip()
    if dataset == 'pubmedqa':
        return text.lower()
    return text.upper().replace('-', '_').replace(' ', '_')

def prediction_answer(record):
    return (record.get('prediction') or {}).get('answer')

def prediction_cited_ids(record):
    prediction = record.get('prediction') or {}
    cited = prediction.get('cited_passage_ids')
    if cited is None:
        cited = (record.get('parsed_answer') or {}).get('cited_passage_ids')
    return cited if isinstance(cited, list) else []

def gold_keys(dataset, record):
    keys = set()
    for item in (record.get('example') or {}).get('gold_evidence', []):
        if dataset == 'pubmedqa':
            keys.add((str(item.get('pubid')), int(item.get('context_idx'))))
        else:
            keys.add((str(item.get('doc_id')), int(item.get('sentence_index'))))
    return keys

def passage_key(dataset, passage):
    metadata = passage.get('metadata') or {}
    if dataset == 'pubmedqa':
        return (str(metadata.get('pubid')), int(metadata.get('context_idx')))
    return (str(metadata.get('doc_id')), int(metadata.get('sentence_index')))

def tag_record(dataset, record):
    retrieved = record.get('retrieved_passages') or []
    retrieved_by_id = {p.get('passage_id'): p for p in retrieved}
    cited_ids = prediction_cited_ids(record)
    invalid_ids = [pid for pid in cited_ids if pid not in retrieved_by_id]
    cited_keys = {passage_key(dataset, retrieved_by_id[pid]) for pid in cited_ids if pid in retrieved_by_id}
    retrieved_keys = {passage_key(dataset, passage) for passage in retrieved}
    gold = gold_keys(dataset, record)
    answer_match = normalize_answer(dataset, prediction_answer(record)) == normalize_answer(dataset, (record.get('example') or {}).get('gold_answer'))
    gold_retrieved = bool(gold & retrieved_keys)
    gold_cited = bool(gold & cited_keys)
    return {
        'dataset': dataset,
        'id': (record.get('example') or {}).get('id'),
        'gold_answer': (record.get('example') or {}).get('gold_answer'),
        'prediction': prediction_answer(record),
        'answer_match': answer_match,
        'generation_error': bool(record.get('generation_error')),
        'retrieved_gold': gold_retrieved,
        'cited_gold': gold_cited,
        'citation_valid': not invalid_ids,
        'no_citation': len(cited_ids) == 0,
        'cited_count': len(cited_ids),
        'invalid_cited_ids': invalid_ids,
        'question': (record.get('example') or {}).get('query'),
    }

case_rows = pd.DataFrame([
    tag_record(dataset, record)
    for dataset, dataset_records in records.items()
    for record in dataset_records
])

display(case_rows.groupby('dataset')[['answer_match', 'generation_error', 'retrieved_gold', 'cited_gold', 'citation_valid', 'no_citation']].mean())
display(case_rows.groupby(['dataset', 'gold_answer', 'prediction']).size().rename('count').reset_index())

## 4. Interesting Cases

The goal here is to learn where errors come from: retrieval miss, generation mistake despite evidence, citation mistake, or model/API parsing failure.

In [ ]:
interesting_filters = {
    'generation_error': case_rows['generation_error'],
    'wrong_answer_but_gold_retrieved': (~case_rows['answer_match']) & case_rows['retrieved_gold'],
    'correct_answer_but_gold_not_cited': case_rows['answer_match'] & case_rows['retrieved_gold'] & (~case_rows['cited_gold']),
    'invalid_citation': ~case_rows['citation_valid'],
    'no_citation': case_rows['no_citation'],
}

interesting_counts = pd.DataFrame([
    {'case_type': name, 'dataset': dataset, 'count': int(mask[case_rows['dataset'] == dataset].sum())}
    for name, mask in interesting_filters.items()
    for dataset in records
])

display(interesting_counts.pivot(index='case_type', columns='dataset', values='count'))

display(case_rows[interesting_filters['wrong_answer_but_gold_retrieved']][[
    'dataset', 'id', 'gold_answer', 'prediction', 'cited_gold', 'citation_valid', 'question'
]].head(10))

## 5. Read One Artifact Row

Use `show_case(dataset, example_id)` to inspect a single row with retrieved passages and citation/gold markers.

In [ ]:
def find_record(dataset, example_id=None, row_index=0):
    if example_id is None:
        return records[dataset][row_index]
    for record in records[dataset]:
        if str((record.get('example') or {}).get('id')) == str(example_id):
            return record
    raise KeyError((dataset, example_id))

def show_case(dataset, example_id=None, row_index=0):
    record = find_record(dataset, example_id=example_id, row_index=row_index)
    tag = tag_record(dataset, record)
    cited_ids = set(prediction_cited_ids(record))
    gold = gold_keys(dataset, record)
    print(f"dataset: {dataset}")
    print(f"id: {tag['id']}")
    print(f"question: {tag['question']}")
    print(f"gold: {tag['gold_answer']} | prediction: {tag['prediction']}")
    print(f"answer_match: {tag['answer_match']} | cited_gold: {tag['cited_gold']} | citation_valid: {tag['citation_valid']}")
    if record.get('generation_error'):
        print('generation_error:', record['generation_error'])
    rows = []
    for passage in record.get('retrieved_passages') or []:
        key = passage_key(dataset, passage)
        metadata = passage.get('metadata') or {}
        rows.append({
            'rank': passage.get('rank'),
            'passage_id': passage.get('passage_id'),
            'cited': passage.get('passage_id') in cited_ids,
            'gold': key in gold,
            'title_or_label': metadata.get('title') or metadata.get('context_label'),
            'text': passage.get('text'),
        })
    display(pd.DataFrame(rows))

show_case('pubmedqa', row_index=0)

## 6. Citation-First Prompt A/B

These files compare the original v1 n=100 outputs against selected v2 citation-first and v3 evidence-summary reruns on the same example IDs. The selected cases intentionally over-sample failures, so the rates below are diagnostic, not full-dataset estimates.

In [ ]:
AB_DIR = PROJECT_ROOT / 'outputs/analysis/generation_ab'

AB_SUMMARIES = {
    'pubmedqa': AB_DIR / 'pubmedqa_v1_vs_v2_vs_v3_debug_ab_summary.json',
    'scifact': AB_DIR / 'scifact_v1_vs_v2_vs_v3_debug_ab_summary.json',
}

AB_COMPARISONS = {
    'pubmedqa': AB_DIR / 'pubmedqa_v1_vs_v2_vs_v3_debug_ab.csv',
    'scifact': AB_DIR / 'scifact_v1_vs_v2_vs_v3_debug_ab.csv',
}

ab_summaries = {
    dataset: load_json(path)
    for dataset, path in AB_SUMMARIES.items()
    if path.exists()
}

display(pd.DataFrame(ab_summaries.values()))

ab_rows = []
for dataset, path in AB_COMPARISONS.items():
    if path.exists():
        frame = pd.read_csv(path)
        frame.insert(0, 'dataset', dataset)
        ab_rows.append(frame)

ab_cases = pd.concat(ab_rows, ignore_index=True) if ab_rows else pd.DataFrame()
ab_preview_columns = [
    'dataset', 'id', 'gold_answer', 'v1_answer', 'v2_answer', 'v3_answer',
    'v1_answer_match', 'v2_answer_match', 'v3_answer_match',
    'v1_cited_gold', 'v2_cited_gold', 'v3_cited_gold',
    'v1_citation_valid', 'v2_citation_valid', 'v3_citation_valid',
    'v3_evidence_summary', 'question'
]
display(ab_cases[[column for column in ab_preview_columns if column in ab_cases.columns]].head(20))

In [ ]:
if not ab_cases.empty:
    display(ab_cases[
        (ab_cases['v1_answer_match'] != ab_cases.get('v3_answer_match', ab_cases['v2_answer_match']))
        | (ab_cases['v1_cited_gold'] != ab_cases.get('v3_cited_gold', ab_cases['v2_cited_gold']))
    ][[
        'dataset', 'id', 'gold_answer', 'v1_answer', 'v2_answer', 'v3_answer',
        'v1_answer_match', 'v2_answer_match', 'v3_answer_match',
        'v1_cited_gold', 'v2_cited_gold', 'v3_cited_gold',
        'v3_evidence_summary', 'question'
    ]])

## 7. V3 Model Check

These files compare the same v3 selected cases between the baseline `qwen3-8b` generation profile and `qwen3.5-flash`. This isolates model choice while keeping retrieval, prompt, and selected IDs fixed.

In [ ]:
MODEL_SUMMARIES = {
    'pubmedqa': AB_DIR / 'pubmedqa_v3_qwen3_8b_vs_qwen35_flash_summary.json',
    'scifact': AB_DIR / 'scifact_v3_qwen3_8b_vs_qwen35_flash_summary.json',
}

MODEL_COMPARISONS = {
    'pubmedqa': AB_DIR / 'pubmedqa_v3_qwen3_8b_vs_qwen35_flash.csv',
    'scifact': AB_DIR / 'scifact_v3_qwen3_8b_vs_qwen35_flash.csv',
}

model_summaries = {
    dataset: load_json(path)
    for dataset, path in MODEL_SUMMARIES.items()
    if path.exists()
}
display(pd.DataFrame(model_summaries.values()))

model_rows = []
for dataset, path in MODEL_COMPARISONS.items():
    if path.exists():
        frame = pd.read_csv(path)
        frame.insert(0, 'dataset', dataset)
        model_rows.append(frame)

model_cases = pd.concat(model_rows, ignore_index=True) if model_rows else pd.DataFrame()
model_preview_columns = [
    'dataset', 'id', 'gold_answer', 'baseline_answer', 'candidate_answer',
    'baseline_answer_match', 'candidate_answer_match',
    'baseline_cited_gold', 'candidate_cited_gold',
    'baseline_citation_valid', 'candidate_citation_valid',
    'candidate_evidence_summary', 'question'
]
display(model_cases[[column for column in model_preview_columns if column in model_cases.columns]].head(20))